<a href="https://colab.research.google.com/github/eoinleen/dianthus-analysis/blob/main/Dianthus_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dianthus TRIC / spectral-shift analysis

Full documentation, example data and version history: https://github.com/eoinleen/dianthus-analysis

**Run order:** settings cell → analysis cell → upload/run cell. After changing a setting, re-run the settings cell and the upload/run cell (the analysis cell only needs re-running if you edit the code).

## Inputs (both kinds can go in one run)
| Layout | What it looks like | How repeats are grouped |
|---|---|---|
| **Wide sheet** | Row 1 = sample names, one per column pair; columns alternate conc. (nM) / signal. A header row under the names is optional. | Column pairs with the same name |
| **Dianthus export table** | The instrument's own table (`Well ID`, `Ligand Conc.`, `Ligand Conc. Unit`, `Ratio` or Fnorm, `Anomaly`) — one file per repeat | File name minus `table-` and trailing `-<number>`s: `table-K11-EL17-3.xlsx` → `K11-EL17`. All export files of one readout are pooled. |

TRIC vs ratio is detected from the signal size, so files don't need naming.

## Settings you will actually change
| Setting | Use |
|---|---|
| `TOTAL_TARGET_NM` | Total tracer in the well (labelled + unlabelled), nM |
| `FIT_MODEL` | `"1:1"` (default) or `"two_site"`. Decide before looking at the data. |
| `TWO_SITE_KD2_NM` | Fix the weak-site Kd (nM) when the top concentration can't define it, e.g. from an independent experiment. `None` = fitted. |
| `MAX_CONC_NM` | Ignore wells above this concentration (all samples; state it as a rule). `None` = use all. |
| `DROP_ANOMALIES` | Export tables only: drop wells the instrument flagged in `Anomaly`. Default `False`. |
| `HILL_FIGURES` | `True` = also write a `hill/` folder: panels, grid and overlays with the Hill fit (EC50 ± CI, n_H) and the classic linearised Hill plot (log[θ/(1−θ)] vs log[L]; slope = n_H, zero crossing = EC50). Hill panels, overlay and Hill plot are appended to the main PPTX after the reported-model slides. Descriptive only; roughly doubles run time. |
| `X_SCI_FROM` | x-axis labels plain below this (nM), 10ⁿ from it. Default 1000 → 0.1 1 10 100 10³ 10⁴. |

## Main figure: `Figure_rows.pdf` / `.svg` / `.png`
One row per readout (TRIC row, then spectral-shift row), two panels per row: fraction bound with the legend ("K11 (K_D = 187 ± 50 nM)" / "n = 5, R² = 0.99"), then the raw signal. It is exactly `PAGE_WIDTH_MM` wide (default 180 mm, an A4 page with margins) and every data box is identical, so **place it at 100%** and the fonts are the sizes set in AESTHETICS. With `HILL_FIGURES = True` there is a matching `Figure_rows_hill`. `Figure_rows.pptx` holds both at true size on A4 slides. The two panels also exist as separate files in each readout folder (`overlay_panel`, `overlay_raw`), same box size.

## Changing figure text
All text on the figures lives in the **LABELS** block of the settings cell: sample display names (`SAMPLE_LABELS`), axis labels, the Kd / EC50 / n_H symbols, the representative-panel title, panel titles on/off, and whether legends show n and R².

## Reading `results.csv`
- `Kd_nM`, `Kd_CI95_low/high`: the reported model (Kd1 for two-site), with residual-bootstrap 95% CI.
- Two-site extras: `Kd2_nM` (+CI if fitted), `frac_tight_site` = fraction of the signal change from the tight site.
- Diagnostics, always run: `dAICc_1to1`, `dAICc_nonspecific`, `dAICc_hill`, `dAICc_twosite` = AICc(reported) − AICc(alternative). **Positive = the alternative fits better; > 6 = strongly.** Also `hill_EC50_nM`, `hill_n` (Dianthus-software-style Hill; with `HILL_FIGURES` also their 95% CIs), and `twosite_Kd1_nM`, `twosite_Kd2_nM`, `twosite_frac_tight`.
- Flags worth acting on: `two-site strongly favoured` (consider `FIT_MODEL = "two_site"` *for the next experiment's pre-specified analysis*), `two-site not resolved by these data` (Kd2 ran beyond the data — fix `TWO_SITE_KD2_NM` or stay with 1:1), `Kd2 near/above top concentration`.

## Changes from v32
Two-site model (reported or diagnostic) · optional Hill figure set incl. linearised Hill plot · methods text states any excluded wells · Hill EC50/n saved · Dianthus export tables read directly · `MAX_CONC_NM` · x-axis 10ⁿ from `X_SCI_FROM` · header rows detected automatically (v32 silently dropped the first data row of sheets without a header row) · `TOTAL_TARGET_NM` re-read on every run · fraction-bound axes widen instead of hiding points; Kd box top-left on fraction-bound panels · overlay legend in Kd order · bad fits no longer carry contradictory flags.


In [ ]:
# ============================================================================
# SETTINGS - edit these per experiment
# ============================================================================

# Readout: "tric" = TRIC Fnorm (%)  |  "ratio" = spectral shift 670/650 ratio.
# The fit is identical; only axis labels, decimals and the methods text change.
READOUT = "tric"

# TOTAL target in the well (labelled + unlabelled), in nM. This is what the
# quadratic needs. It is NOT the dye concentration.
#   total = dye_in_well / labelled_fraction
#   e.g. K11 tracer: 10 nM dye / 0.64 = 15.6 nM;  K33: 10 / 0.56 = 17.7 nM
# One value per run, so analyse tracers with different totals in separate files.
TOTAL_TARGET_NM = 10
TRACER_DYE_NM   = 10      # reference only (reported, not used in the fit)

# Free-text descriptions used in the auto-generated methods paragraph.
TRACER_DESCRIPTION = ("the labelled target protein "
                      "(edit: protein, dye and labelling kit)")
TITRANT_NAME = "titrant"   # name of the titrated protein: methods text and the default x-axis label

# Input layouts (both can be fed in the same run):
#  (a) Wide sheet: row 1 = sample names (one per column pair); columns alternate
#      concentration (nM) / signal (Fnorm per mil for TRIC, 670/650 ratio for spectral
#      shift). A column-header row under the names is optional: text rows are skipped.
#      Column pairs that share a sample name are treated as repeats of that sample.
#  (b) Dianthus software export table (columns "Well ID", "Ligand Conc.", "Ligand Conc.
#      Unit", "Ratio"/Fnorm, ...): one file = one repeat, units converted to nM. The
#      sample name comes from the file name with a leading "table-" and trailing
#      -<number> groups removed (table-K11-EL17-3.xlsx -> "K11-EL17"), and all export
#      files of the same readout are pooled, so repeats are fitted together.
DATA_START_ROW = 1          # 0-based row where data may start in wide sheets (leave at 1)
DROP_ANOMALIES = False      # export tables: True = drop wells with an entry in "Anomaly"
                            #   (e.g. "Y-Scan Failed"). Default keeps every well.
MAX_CONC_NM = None          # None = use every concentration; e.g. 15000 = ignore wells above
                            #   15 uM (applies to every sample - state it as a rule)

# ============================================================================
# MODEL - choose before looking at the data. The other models always run as
# diagnostics (dAICc columns in results.csv; > 6 = strongly better fit).
# ============================================================================
FIT_MODEL = "1:1"          # "1:1"      = single site with ligand depletion (Morrison)
                           # "two_site" = two independent classes of site: reports Kd1
                           #   (tight site, with bootstrap CI), Kd2 and the fraction of the
                           #   signal change from the tight site
HILL_FIGURES = False       # True = also write a hill/ folder: panels, grid and overlays with
                           #   the Hill fit (EC50 ± CI, n_H) plus the classic linearised Hill plot
                           #   (log[θ/(1-θ)] vs log[L], slope = n_H). Descriptive only - the
                           #   reported Kd is unchanged. Roughly doubles run time (own bootstrap).
TWO_SITE_KD2_NM = None     # None = fit Kd2; a number (nM) = fix Kd2, e.g. from an
                           #   independent experiment, when the top concentration is too
                           #   low to define it (results flag "Kd2 near/above top conc.")

PLOT_NORMALISED = False    # False = raw signal (Fnorm or ratio); True = fraction bound (0-1),
                           #   each repeat scaled by its own fitted baselines
SHOW_RESIDUALS = False     # True = add residual panels under each fit

# Which fits count as determined (curve drawn, Kd reported) - same fit either way.
FIT_CRITERIA = "screen"    # "screen" = v17 rules, for initial screens:
                           #     R2 < SCREEN_MIN_R2 -> N.D.; R2 < SCREEN_APPROX_R2 or <50% bound
                           #     at top conc -> "Kd ~"; sorted R2 >= 0.75 by Kd, then the rest by R2
                           # "strict" = for anything beyond a screen:
                           #     N.D. if signal/noise < MIN_SIGNAL_TO_NOISE, or the 95% CI spans
                           #     more than MAX_CI_FOLD-fold, or the bootstrap failed
SCREEN_MIN_R2 = 0.5
SCREEN_APPROX_R2 = 0.88
MIN_SIGNAL_TO_NOISE = 5
MAX_CI_FOLD = 10
KD_LABEL = "plus_minus"    # "plus_minus" = v17 "Kd = x ± y" (y = half the 95% CI width)
                           # "ci" = "Kd = x (low–high)", exact when the CI is asymmetric
Y_DECIMALS = 0 if READOUT == "tric" else 2   # decimals on the raw-signal axis (v17 ratio: 2)
EXPORT_PPTX = True         # v17 PowerPoint figure: 3 x 6 panels per portrait slide
PPTX_FILENAME = "Figure_TRIC.pptx" if READOUT == "tric" else "Figure_SpectralShift.pptx"
BAD_FIT_PLOT = "data_only" # "data_only" = points, no curve | "skip" = leave out of figures
                           # | "plot" = draw the curve anyway (still flagged N.D. in the table)

# Overlay: all chosen samples on one plot. Columns sharing a row-1 name are repeats;
# points = mean +/- 1 SD across repeats at each concentration (display only - the
# fit always uses every individual point). Points measured in only one repeat are
# drawn hollow, without an error bar.
OVERLAY = True
OVERLAY_AXIS = "normalised"   # "normalised" = fraction bound, each repeat scaled by its own
                              #   fitted baselines (recommended)
                              # "raw" = mean raw signal; SD then includes baseline offsets
                              #   between repeats
OVERLAY_SAMPLES = None        # None = all determined samples, sorted by Kd
                              # or a list in legend order, e.g. ["EL17", "EL17_R2"]
OVERLAY_CI_BAND = True        # shaded 95% bootstrap band on each fit
OVERLAY_KD_TICKS = True       # short tick on the x-axis at each sample's Kd
OVERLAY_LEGEND_NORM_ONLY = True  # Kd legend only on fraction-bound overlays; raw and
                              #   representative panels keep the Kd notch but drop the legend
                              #   text. False = legend (with Kd) on every overlay, as before.
OVERLAY_LEGEND_DETAIL = True  # legend entry "K11 (K_D = 187 ± 50 nM)" with "n = 5, R² = 0.991"
                              #   below (Hill: EC50 ± ..., n_H). False = "K11 (K_D = 187 nM, n=5)"
OVERLAY_FILENAME = "overlay"
OVERLAY_RAW_PANEL = True      # also save the row panels as separate files (overlay_panel = fraction
                              #   bound, overlay_raw), same box size as in Figure_rows

# ============================================================================
# LABELS - every piece of text on the figures. Matplotlib mathtext works inside $...$
# (e.g. r"$K_\mathrm{D}$"); use r"..." strings when there is a backslash.
# ============================================================================
SAMPLE_LABELS = {}         # display names, e.g. {"K11-EL17": "K11", "K33-EL17": "K33"}. Applied on
                           #   loading, so results.csv and OVERLAY_SAMPLES use the new names too;
                           #   two samples given the same name are pooled as repeats.
X_LABEL           = f"[{TITRANT_NAME}] (nM)"   # or any text, e.g. "[EL17] (nM)"
Y_LABEL_FRACTION  = "Fraction bound"
Y_LABEL_RATIO     = "Ratio"
Y_LABEL_TRIC      = r"$F_\mathrm{norm}$ (‰)"
Y_LABEL_HILL_PLOT = r"log$_{10}$[θ / (1 − θ)]"
KD_SYMBOL         = r"$K_\mathrm{D}$"
KD1_SYMBOL        = r"$K_\mathrm{D1}$"     # two-site model, tight site
KD2_SYMBOL        = r"$K_\mathrm{D2}$"     # two-site model, weak site
EC50_SYMBOL       = r"EC$_{50}$"
NH_SYMBOL         = r"$n_\mathrm{H}$"
SHOW_PANEL_TITLES = True   # sample name above each per-sample panel
LEGEND_SHOW_N     = True   # "n = 4" (number of repeats) in legends
LEGEND_SHOW_R2    = True   # "R² = ..." in legends and per-sample text boxes

# ============================================================================
# FIGURE ROWS (D/E style) - Figure_rows.pdf/svg/png: one row per readout (TRIC row, then
# ratio row), panels left to right from ROW_PANELS. Sized to PAGE_WIDTH_MM exactly:
# place at 100% and every font is the size set below. All boxes are identical.
# ============================================================================
PAGE_WIDTH_MM  = 180       # total figure width (A4 with ~15 mm margins = 180)
ROW_PANELS     = ("normalised", "raw")   # per row; "normalised" = fraction bound (with legend)
ROW_BOX_ASPECT = 1.45      # data-box width / height
ROW_MARGIN_L   = 0.62      # inches around each box, for y-label + ticks ...
ROW_MARGIN_R   = 0.12
ROW_MARGIN_B   = 0.45      # ... x-label + ticks
ROW_MARGIN_T   = 0.10

# ============================================================================
# AESTHETICS - figure look. Sizes in points; edit here, not in the code.
# One set of font sizes is used by EVERY output (panels, rows, overlays, Hill plot).
# ============================================================================
X_SCI_FROM       = 1000    # x-axis tick labels: plain numbers below this (nM), 10^n from it
                           #   (0.1 1 10 100 10³ 10⁴); 1e9 = plain numbers throughout
FONT_FAMILY      = "DejaVu Sans"   # any installed font, e.g. "Arial" (warns + falls back if absent)
FONT_SIZE        = 9        # axis labels and the Kd/R2 box on per-sample panels
TITLE_FONT_SIZE  = 10      # sample name above each per-sample panel
TICK_FONT_SIZE   = 8       # axis tick numbers
LEGEND_FONT_SIZE = 7       # legends
MARKER_PT        = 4       # overlay marker size (points)
PANEL_W, PANEL_H = 2.4, 1.75   # one panel, inches (also sets the PPTX grid)
MARKER_SIZE      = 25      # data points in the per-sample panels (scatter area, pts^2)
MARKER_COLOR     = "#8B0000"
FIG_DPI          = 600     # PNG export resolution for panels and overlay
TRANSPARENT      = True    # transparent panel backgrounds in every PNG/SVG (the black box and
                           #   axes stay); hollow markers become truly open. The Kd/R2 box keeps
                           #   its white pad for legibility. False = white backgrounds.
PANEL_BOX_W, PANEL_BOX_H = 1.9, 1.35   # fixed DATA-BOX size (inches) for single exported panels
                               #   (Binder_*.png and the panel-sized overlays): the image grows
                               #   around the box to fit labels, so stacked boxes line up whatever
                               #   the y-label width. PANEL_W/PANEL_H above still set grid + PPTX tiles.
PANEL_MARGIN_L   = 0.62    # inches reserved for the y-axis label + numbers (fixed, so every
PANEL_MARGIN_B   = 0.42    #   panel is the SAME overall size with the box in the SAME place -
PANEL_MARGIN_R   = 0.10    #   boxes then line up across panels whatever the y-label width.
PANEL_MARGIN_T   = 0.26    #   Widen L/T if a long y-label or title ever clips.

N_BOOTSTRAP = 500
RANDOM_SEED = 1
# Multiple input files (TRIC and spectral-shift) are analysed together. Each file's
# readout is detected from the signal size - ratio values sit near 1, Fnorm near 900 per
# mil, so the two never overlap and you don't have to name the files. Feed both.
AUTO_READOUT = True       # False = treat every file as the READOUT set at the top
ROWS_PPTX = "Figure_rows.pptx"   # Figure_rows (and _hill) at true size on A4 slides

OUTPUT_DIR = "dianthus_results"

In [ ]:

# ============================================================================
# Dianthus TRIC / spectral shift analysis v33
#   v33: FIT_MODEL switch ("1:1" or "two_site"); Hill EC50 + n and a two-site fit
#        saved as diagnostics for every sample; Dianthus software export tables
#        read directly (one table = one repeat); MAX_CONC_NM cap; x-axis switches
#        to 10^n from X_SCI_FROM; header rows detected automatically; T re-read
#        on every run; fraction-bound axes widen instead of hiding points.
# ----------------------------------------------------------------------------
# The reported model is chosen BEFORE looking at the data (FIT_MODEL). Repeats
# of a sample are fitted globally with shared affinity parameters and their own
# baselines. The other models are always run as diagnostics (dAICc columns)
# and never replace the reported values.
# ============================================================================

import os, re, shutil, sys

NOTEBOOK_VERSION = "v33"     # bump with each change; tag the GitHub release with the same name
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

T = float(TOTAL_TARGET_NM)
def set_readout(r):
    """Point the readout-dependent labels at TRIC (Fnorm, per mil) or spectral shift (ratio)."""
    global READOUT, Y_RAW, Y_DECIMALS, PPTX_FILENAME
    assert r in ("tric", "ratio"), "readout must be 'tric' or 'ratio'"
    READOUT = r
    Y_RAW = Y_LABEL_TRIC if r == "tric" else Y_LABEL_RATIO        # raw-signal axis label (LABELS)
    Y_DECIMALS = 0 if r == "tric" else 2
    PPTX_FILENAME = "Figure_TRIC.pptx" if r == "tric" else "Figure_SpectralShift.pptx"

set_readout(READOUT)
rng = np.random.default_rng(RANDOM_SEED)

# ----------------------------------------------------------------------------
# Models (concentrations in nM throughout; Kd fitted as log10 for conditioning)
# ----------------------------------------------------------------------------

def frac_bound(c, kd, t=None):
    """Morrison quadratic, written in the cancellation-free form
    f = 2c / (b + sqrt(b^2 - 4ct)),  b = c + t + kd  (identical algebraically).
    t defaults to the CURRENT global T (read at call time, not at definition)."""
    t = T if t is None else t
    c = np.asarray(c, float)
    b = c + t + kd
    return 2 * c / (b + np.sqrt(np.maximum(b * b - 4 * c * t, 0)))


def frac_bound_hill(c, kd, n, t=None):
    """Hill binding with depletion, mass balance solved by bisection for free
    ligand (L_tot = L_free + t * theta(L_free)). kd here is the EC50."""
    t = T if t is None else t
    c = np.asarray(c, float)
    lo, hi = np.zeros_like(c), c.copy()
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        theta = 1 / (1 + (kd / np.maximum(mid, 1e-30)) ** n)
        over = mid + t * theta > c
        hi, lo = np.where(over, mid, hi), np.where(over, lo, mid)
    lf = 0.5 * (lo + hi)
    return 1 / (1 + (kd / np.maximum(lf, 1e-30)) ** n)


# Each model: shared params (after logKd) and per-repeat params.
# twosite: two independent classes of site, f = f1*f(Kd1) + (1-f1)*f(Kd2), Kd2 > Kd1.
#   Shared "dlog" = log10(Kd2/Kd1) (kept >= 0.5, so the sites stay >3-fold apart and
#   cannot swap labels); "f1" = fraction of the signal change from the tight site.
#   With TWO_SITE_KD2_NM set, Kd2 is fixed and only f1 is shared.
MODELS = {
    "basic": dict(shared=[],    per=["R_unbound", "R_bound"]),
    "nonspecific": dict(shared=[], per=["R_unbound", "R_bound", "ns_slope"]),
    "hill": dict(shared=["n"],  per=["R_unbound", "R_bound"]),
    "twosite": dict(shared=["dlog", "f1"], per=["R_unbound", "R_bound"]),
}
MODEL_KEY = {"1:1": "basic", "two_site": "twosite"}


def _configure_models():
    MODELS["twosite"]["shared"] = ["f1"] if TWO_SITE_KD2_NM else ["dlog", "f1"]


def kd2_of(p):
    """Kd2 (nM) from a twosite parameter vector."""
    return float(TWO_SITE_KD2_NM) if TWO_SITE_KD2_NM else 10 ** (p[0] + p[1])


def model_frac(model, c, p):
    """Fraction of the signal change (0 -> 1) for parameter vector p."""
    kd = 10 ** p[0]
    if model == "hill":
        return frac_bound_hill(c, kd, p[1])
    if model == "twosite":
        f1 = p[len(MODELS["twosite"]["shared"])]       # f1 is the last shared parameter
        return f1 * frac_bound(c, kd) + (1 - f1) * frac_bound(c, kd2_of(p))
    return frac_bound(c, kd)


def baselines(model, p, n_rep):
    """Per-repeat (R_unbound, R_bound) arrays."""
    m = MODELS[model]
    ns, npr = len(m["shared"]), len(m["per"])
    per = p[1 + ns:].reshape(n_rep, npr)
    return per[:, 0], per[:, 1]


def _predict(model, c, rep, p, n_rep):
    m = MODELS[model]
    ns, npr = len(m["shared"]), len(m["per"])
    per = p[1 + ns:].reshape(n_rep, npr)[rep]
    ru, rb = per[:, 0], per[:, 1]
    y = ru + model_frac(model, c, p) * (rb - ru)
    if model == "nonspecific":
        y = y + per[:, 2] * c
    return y


def _shared_bounds(model):
    """(start, lower, upper) for the shared parameters of each model."""
    if model == "hill":
        return [1.0], [0.3], [5.0]
    if model == "twosite":
        if TWO_SITE_KD2_NM:
            return [0.7], [0.0], [1.0]
        return [1.5, 0.7], [0.5, 0.0], [5.0, 1.0]
    return [], [], []


def global_fit(model, c, y, rep, n_rep, p0=None):
    """Fit all repeats of one sample at once. Returns params, residuals."""
    m = MODELS[model]
    cpos = c[c > 0]
    lo_kd, hi_kd = np.log10(cpos.min() / 100), np.log10(cpos.max() * 100)
    s0, slb, sub = _shared_bounds(model)
    starts = [p0] if p0 is not None else None
    if p0 is None:
        # Baseline guesses from the lowest / highest concentrations of each repeat
        # (data are sorted, so order on the plate does not matter).
        per0 = []
        for r in range(n_rep):
            yr = y[rep == r]
            g = [np.mean(yr[:2]), np.mean(yr[-2:])]
            if model == "nonspecific":
                g.append(0.0)
            per0 += g
        starts = [[np.log10(np.median(cpos))] + s0 + per0]
        if model == "twosite":                        # a few starts: the surface is bumpy
            starts = []
            for lk in np.log10(np.median(cpos)) + np.array([-1.0, 0.0]):
                for dl in ([1.0, 2.0, 3.0] if not TWO_SITE_KD2_NM else [None]):
                    for f1 in (0.5, 0.8):
                        sh = [f1] if TWO_SITE_KD2_NM else [dl, f1]
                        starts.append([lk] + sh + per0)
    npar = len(starts[0])
    lb = [lo_kd] + slb + [-np.inf] * (npar - 1 - len(m["shared"]))
    ub = [hi_kd] + sub + [np.inf] * (npar - 1 - len(m["shared"]))
    if model == "nonspecific":
        # Bound the slope so it cannot impersonate the binding transition:
        # at the top concentration it may contribute at most the full amplitude.
        amp = np.ptp(y)
        for r in range(n_rep):
            i = 1 + 3 * r + 2
            lb[i], ub[i] = -amp / cpos.max(), amp / cpos.max()
    f = lambda cc, *p: _predict(model, cc, rep, np.array(p), n_rep)
    best = None
    for st in starts:
        st = np.clip(st, np.array(lb) + 1e-9, np.array(ub) - 1e-9)
        try:
            p, _ = curve_fit(f, c, y, p0=st, bounds=(lb, ub), maxfev=20000)
        except Exception:
            continue
        rss = np.sum((y - f(c, *p)) ** 2)
        if best is None or rss < best[0]:
            best = (rss, p)
    if best is None:
        raise RuntimeError(f"{model} fit failed")
    p = best[1]
    return p, y - f(c, *p)


def aicc(rss, n, k):
    if n - k - 1 <= 0 or rss <= 0:
        return np.inf
    return n * np.log(rss / n) + 2 * k + 2 * k * (k + 1) / (n - k - 1)


def bootstrap_kd(model, c, y, rep, n_rep, p_hat, resid):
    """Residual bootstrap of the reported global fit. Residuals are resampled
    within each repeat, so each repeat keeps its own noise level.
    Returns CI of Kd (Kd1 for two-site) and the bootstrap parameter vectors."""
    fitted = y - resid
    pbs = []
    for _ in range(N_BOOTSTRAP):
        yb = fitted.copy()
        for r in range(n_rep):
            idx = np.where(rep == r)[0]
            yb[idx] += rng.choice(resid[idx], idx.size, replace=True)
        try:
            pb, _ = global_fit(model, c, yb, rep, n_rep, p0=p_hat)
            pbs.append(pb)
        except Exception:
            pass
    pbs = np.array(pbs)
    if len(pbs) < 0.8 * N_BOOTSTRAP:
        return np.nan, np.nan, pbs
    return *np.percentile(10 ** pbs[:, 0], [2.5, 97.5]), pbs


# ----------------------------------------------------------------------------
# Loading
# ----------------------------------------------------------------------------

def detect_readout(samples):
    """ratio (~1) vs TRIC Fnorm (~900 per mil) from the median signal; disjoint ranges."""
    ys = [r[1] for reps in samples.values() for r in reps]
    med = np.nanmedian(np.concatenate(ys)) if ys else 1.0
    return "ratio" if med < 10 else "tric"


_DROPPED = []                                         # anomaly labels of dropped wells (methods text)
_UNIT_TO_NM = {"pm": 1e-3, "nm": 1.0, "um": 1e3, "µm": 1e3, "μm": 1e3, "mm": 1e6}


def _read_table(path):
    return (pd.read_csv(path, header=None) if path.lower().endswith(".csv")
            else pd.read_excel(path, header=None))


def is_dianthus_export(path):
    raw = _read_table(path)
    return any(str(v).strip() == "Ligand Conc." for v in raw.iloc[0])


def export_sample_name(path):
    """table-K11-EL17-3.xlsx -> 'K11-EL17': drop a leading 'table-' and any trailing
    -<number> groups (repeat indices), so repeats of one sample share a name."""
    stem = os.path.splitext(os.path.basename(path))[0]
    stem = re.sub(r"^table[-_ ]", "", stem, flags=re.I)
    return re.sub(r"([-_ ]\d+)+$", "", stem) or stem


def _cap(c, y):
    ok = np.isfinite(c) & np.isfinite(y) & (c >= 0)       # drop a row if EITHER is missing
    if MAX_CONC_NM is not None:
        ok &= c <= MAX_CONC_NM
    c, y = c[ok], y[ok]
    order = np.argsort(c)                                 # plate order irrelevant
    return c[order], y[order]


def load_export(path):
    """One Dianthus software export table = one repeat. Columns used: 'Ligand Conc.',
    'Ligand Conc. Unit', and 'Ratio' (spectral shift) or the Fnorm column (TRIC).
    Wells with an entry in 'Anomaly' are dropped only if DROP_ANOMALIES is True."""
    d = pd.read_csv(path) if path.lower().endswith(".csv") else pd.read_excel(path)
    cols = {str(k).strip(): k for k in d.columns}
    sig = cols.get("Ratio") or next((k for s, k in cols.items() if "norm" in s.lower()), None)
    if sig is None:
        raise ValueError(f"{path}: no 'Ratio' or Fnorm column")
    c = pd.to_numeric(d[cols["Ligand Conc."]], errors="coerce").to_numpy(float)
    if "Ligand Conc. Unit" in cols:
        unit = d[cols["Ligand Conc. Unit"]].astype(str).str.strip().str.lower()
        c = c * unit.map(lambda u: _UNIT_TO_NM.get(u, np.nan)).to_numpy(float)
    y = pd.to_numeric(d[sig], errors="coerce").to_numpy(float)
    if DROP_ANOMALIES and "Anomaly" in cols:
        bad = d[cols["Anomaly"]].astype(str).str.strip().isin(["-", "", "nan"]) == False
        if bad.any():
            print(f"  {os.path.basename(path)}: dropping {bad.sum()} flagged well(s)")
            _DROPPED.extend(d.loc[bad, cols["Anomaly"]].astype(str).str.strip())
        y = np.where(bad, np.nan, y)
    return {export_sample_name(path): [_cap(c, y)]}


def load(path):
    if is_dianthus_export(path):
        return load_export(path)
    raw = _read_table(path)
    samples = {}
    for i in range(raw.shape[1] // 2):
        name = raw.iat[0, 2 * i]
        if pd.isna(name):
            name = raw.iat[0, 2 * i + 1]
        name = str(name).strip().strip('"\'') if pd.notna(name) else f"Sample {i+1}"
        # Everything below the name row; text header rows convert to NaN and drop out.
        c = pd.to_numeric(raw.iloc[DATA_START_ROW:, 2 * i], errors="coerce").to_numpy()
        y = pd.to_numeric(raw.iloc[DATA_START_ROW:, 2 * i + 1], errors="coerce").to_numpy()
        samples.setdefault(name, []).append(_cap(c, y))
    return samples


# ----------------------------------------------------------------------------
# Per-sample analysis
# ----------------------------------------------------------------------------

def analyse_sample(name, repeats):
    RM = MODEL_KEY[FIT_MODEL]
    n_rep = len(repeats)
    c = np.concatenate([r[0] for r in repeats])
    y = np.concatenate([r[1] for r in repeats])
    rep = np.concatenate([np.full(len(r[0]), i) for i, r in enumerate(repeats)])

    out = dict(sample=name, n_repeats=n_rep, n_points=len(c), total_target_nM=T, model=FIT_MODEL)

    p, resid = global_fit(RM, c, y, rep, n_rep)
    kd = 10 ** p[0]
    lo, hi, pbs = bootstrap_kd(RM, c, y, rep, n_rep, p, resid)
    out.update(Kd_nM=kd, Kd_CI95_low=lo, Kd_CI95_high=hi)
    if RM == "twosite":
        out["Kd2_nM"] = kd2_of(p)
        out["Kd2_fixed"] = bool(TWO_SITE_KD2_NM)
        out["frac_tight_site"] = p[len(MODELS["twosite"]["shared"])]
        if len(pbs) and not TWO_SITE_KD2_NM:
            k2 = [kd2_of(pb) for pb in pbs]
            out["Kd2_CI95_low"], out["Kd2_CI95_high"] = np.percentile(k2, [2.5, 97.5])

    # Each repeat on its own (same model): the spread is the honest error bar.
    kd_each = []
    for r in range(n_rep):
        try:
            pr, _ = global_fit(RM, c[rep == r], y[rep == r], np.zeros((rep == r).sum(), int), 1)
            kd_each.append(10 ** pr[0])
        except Exception:
            kd_each.append(np.nan)
    kd_each = np.array(kd_each)
    out["Kd_each_repeat_nM"] = "; ".join(f"{k:.3g}" for k in kd_each)
    if n_rep >= 2:
        out["Kd_repeats_mean_nM"] = np.nanmean(kd_each)
        out["Kd_repeats_SD_nM"] = np.nanstd(kd_each, ddof=1)

    # Quality numbers
    ru, rb = baselines(RM, p, n_rep)
    amp = np.abs(rb - ru)                           # |R_bound - R_unbound| per repeat
    noise = np.std(resid, ddof=len(p))
    out["signal_to_noise"] = np.mean(amp) / noise if noise > 0 else np.inf
    out["frac_bound_at_top"] = float(model_frac(RM, c.max(), p))
    # SS_tot about each repeat's own mean, so offsets between repeats cannot inflate R2
    ss_tot = sum(np.sum((y[rep == r] - y[rep == r].mean()) ** 2) for r in range(n_rep))
    out["R2"] = 1 - np.sum(resid ** 2) / ss_tot

    # Diagnostics: do other models fit clearly better?  (never change the reported Kd)
    # dAICc_X = AICc(reported) - AICc(X): positive = X fits better; > 6 is strong.
    a_rep = aicc(np.sum(resid ** 2), len(c), len(p))
    hill_fit = None
    for alt in ("basic", "nonspecific", "hill", "twosite"):
        if alt == RM:
            continue
        try:
            pa, ra = global_fit(alt, c, y, rep, n_rep)
            out[f"dAICc_{'1to1' if alt == 'basic' else alt}"] = a_rep - aicc(np.sum(ra ** 2), len(c), len(pa))
            if alt == "hill":
                out["hill_EC50_nM"], out["hill_n"] = 10 ** pa[0], pa[1]
                hill_fit = (pa, ra)
            if alt == "twosite":
                out["twosite_Kd1_nM"], out["twosite_Kd2_nM"] = 10 ** pa[0], kd2_of(pa)
                out["twosite_frac_tight"] = pa[len(MODELS["twosite"]["shared"])]
        except Exception:
            out[f"dAICc_{'1to1' if alt == 'basic' else alt}"] = np.nan

    flags = []
    if kd < T / 2:
        flags.append(f"Kd below resolution limit (~{T/2:.3g} nM): reported as < upper 95% CL")
    if out["frac_bound_at_top"] < 0.8:
        flags.append(f"not saturated ({100*out['frac_bound_at_top']:.0f}% bound at top conc.)")
    if c.max() < 3 * kd:
        flags.append("top concentration < 3x Kd")
    if out["signal_to_noise"] < 5:
        flags.append("low signal/noise")
    if out.get("dAICc_nonspecific", 0) > 6:
        flags.append("non-specific term strongly favoured: inspect")
    if out.get("dAICc_hill", 0) > 6:
        flags.append(f"Hill strongly favoured (n={out.get('hill_n', np.nan):.2f}): inspect")
    if out.get("dAICc_twosite", 0) > 6:
        flags.append("two-site strongly favoured: inspect")
    if RM == "twosite" and not TWO_SITE_KD2_NM and out["Kd2_nM"] > 0.5 * c.max():
        flags.append("Kd2 near/above top concentration: poorly determined (consider TWO_SITE_KD2_NM)")
    if n_rep >= 2 and np.nanmax(kd_each) / np.nanmin(kd_each) > 3:
        flags.append("repeats disagree >3-fold")
    if np.isnan(lo):
        flags.append("bootstrap unstable")

    at_cap = kd > 0.99 * c.max() * 100               # ran into the upper Kd bound: no real fit
    # A free two-site fit whose weak site lies far beyond the data, or whose tight site
    # carries almost none of the signal, has collapsed into "1:1 + a line": not resolved.
    unresolved = (RM == "twosite" and not TWO_SITE_KD2_NM and
                  (out["Kd2_nM"] > 10 * c.max() or not 0.05 < out["frac_tight_site"] < 0.95))
    if unresolved:
        flags = [f for f in flags if not f.startswith("Kd2 near")]
        flags.insert(0, "two-site not resolved by these data: fix Kd2 (TWO_SITE_KD2_NM) or use 1:1")
    if FIT_CRITERIA == "strict":
        bad = (at_cap or unresolved or out["signal_to_noise"] < MIN_SIGNAL_TO_NOISE or np.isnan(lo)
               or hi / lo > MAX_CI_FOLD)
        approx = False
    else:                                            # v17 screen rules
        bad = at_cap or unresolved or out["R2"] < SCREEN_MIN_R2
        approx = not bad and (out["R2"] < SCREEN_APPROX_R2 or out["frac_bound_at_top"] < 0.5)
    out["Kd_determined"] = not bad
    out["Kd_approx"] = approx
    if bad:                                          # the other flags describe a fit that isn't there
        flags = ["BAD FIT: Kd not determined"] + [f for f in flags if f.startswith(("low", "repeats", "two-site not"))]
    elif approx:
        flags.insert(0, "approximate Kd (screen criteria)")
    out["flags"] = " | ".join(flags) if flags else "none"

    plot_data = dict(c=c, y=y, rep=rep, p=p, resid=resid, kd=kd, n_rep=n_rep, kd_each=kd_each,
                     model=RM, ru=ru, rb=rb, boot=pbs if np.isfinite(lo) else None)

    # Hill set (descriptive only): EC50 with its own bootstrap CI, for the hill/ figures.
    if HILL_FIGURES and hill_fit is not None:
        ph, rh = hill_fit
        hlo, hhi, hboot = bootstrap_kd("hill", c, y, rep, n_rep, ph, rh)
        out["hill_EC50_CI95_low"], out["hill_EC50_CI95_high"] = hlo, hhi
        out["hill_n_CI95_low"], out["hill_n_CI95_high"] = (
            np.percentile(hboot[:, 1], [2.5, 97.5]) if len(hboot) else (np.nan, np.nan))
        hru, hrb = baselines("hill", ph, n_rep)
        plot_data["hill"] = dict(p=ph, resid=rh, ru=hru, rb=hrb, R2=1 - np.sum(rh ** 2) / ss_tot,
                                 boot=hboot if np.isfinite(hlo) else None)
    return out, plot_data


# ----------------------------------------------------------------------------
# Plotting - v17 panel style
# ----------------------------------------------------------------------------

from io import BytesIO
from matplotlib.ticker import FuncFormatter, FormatStrFormatter, LogLocator, NullFormatter, MultipleLocator, FixedLocator, MaxNLocator

plt.rcParams.update({"font.family": FONT_FAMILY, "font.size": FONT_SIZE,
                     "savefig.transparent": TRANSPARENT,
                     "svg.fonttype": "none", "pdf.fonttype": 42})
# fraction-bound ticks every 0.2, never labelled above 1.0 (headroom for the Kd box stays blank)
_FRAC_TICKS = FixedLocator(np.round(np.arange(-1.0, 1.01, 0.2), 1) + 0.0)   # +0.0: no "-0.0"
_SUP = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")


def _v17_log_label(x, pos):                 # plain numbers below X_SCI_FROM, then 10^n
    if x <= 0:
        return ""
    if x < X_SCI_FROM:
        return f"{int(round(x))}" if x >= 1 else f"{x:g}"
    e = int(round(np.log10(x)))
    m = x / 10 ** e
    sup = str(e).translate(_SUP)
    return f"10{sup}" if abs(m - 1) < 0.01 else f"{m:.0f}×10{sup}"


def _decades(a):
    """A labelled major tick at every decade, minor ticks between (v17 look)."""
    a.xaxis.set_major_locator(LogLocator(base=10, numticks=30))
    a.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10), numticks=30))
    a.xaxis.set_major_formatter(FuncFormatter(_v17_log_label))
    a.xaxis.set_minor_formatter(NullFormatter())


def _r2_line(res):
    return f"\nR² = {res['R2']:.3f}" if LEGEND_SHOW_R2 else ""


def _kd_text(res):
    """Text box on per-sample panels. Symbols and optional lines come from LABELS."""
    if res.get("model") == "hill":                    # Hill set: EC50 and n_H, no Kd
        e, lo, hi = res["Kd_nM"], res["Kd_CI95_low"], res["Kd_CI95_high"]
        return f"{EC50_SYMBOL} = {_val_pm(e, lo, hi)}\n{NH_SYMBOL} = {res['hill_n']:.2f}{_r2_line(res)}"
    if not res["Kd_determined"]:
        return f"{KD_SYMBOL} = N.D."
    D = KD1_SYMBOL if res.get("model") == "two_site" else KD_SYMBOL
    if res["Kd_nM"] < T / 2:                 # below the resolution limit
        ub = res["Kd_CI95_high"] if np.isfinite(res["Kd_CI95_high"]) else T / 2
        return f"{D} < {ub:.1f} nM{_kd2_line(res)}{_r2_line(res)}"
    kd, lo, hi = res["Kd_nM"], res["Kd_CI95_low"], res["Kd_CI95_high"]
    sym = "~" if res.get("Kd_approx") else "="
    if KD_LABEL == "ci" and np.isfinite(lo):
        s = f"{D} {sym} {kd:.1f} nM ({lo:.1f}–{hi:.1f})"
    elif np.isfinite(lo):                    # v17: half-width of the 95% CI
        s = f"{D} {sym} {kd:.1f} ± {(hi - lo) / 2:.1f} nM"
    else:
        s = f"{D} {sym} {kd:.1f} nM"
    return f"{s}{_kd2_line(res)}{_r2_line(res)}"


def _kd2_line(res):
    """Second line for two-site fits: the weak site's Kd (in µM when >= 1000 nM)."""
    if res.get("model") != "two_site" or not np.isfinite(res.get("Kd2_nM", np.nan)):
        return ""
    k2 = res["Kd2_nM"]
    v = f"{k2 / 1000:.2g} µM" if k2 >= 1000 else f"{k2:.0f} nM"
    return f"\n{KD2_SYMBOL} {'(fixed) ' if res.get('Kd2_fixed') else ''}= {v}"


def _norm_ylim(ax, vals, pad_lo=-0.1, pad_hi=1.1):
    """Fraction-bound axis: default -0.1..1.1, widened so no point is ever hidden."""
    v = np.asarray(vals, float)
    v = v[np.isfinite(v)]
    lo = min(pad_lo, v.min() - 0.05) if v.size else pad_lo
    hi = max(pad_hi, v.max() + 0.05) if v.size else pad_hi
    ax.set_ylim(lo, hi)


def draw(ax, ax_res, name, d, res):
    c, y, rep, p, n_rep = d["c"], d["y"], d["rep"], d["p"], d["n_rep"]
    ru_r, rb_r = d["ru"], d["rb"]                       # per-repeat fitted baselines
    amp = rb_r - ru_r
    show_fit = res["Kd_determined"] or BAD_FIT_PLOT == "plot"
    norm = PLOT_NORMALISED and show_fit                # bad fit: always raw
    yy = (y - ru_r[rep]) / amp[rep] if norm else y
    pos = c > 0
    ax.scatter(c[pos], yy[pos], s=MARKER_SIZE, color=MARKER_COLOR, alpha=0.7, marker="x",
               linewidths=1, zorder=3)
    px, py = [c[pos]], [yy[pos]]                       # everything the label must avoid
    if show_fit:
        xs = np.logspace(np.log10(c[pos].min()), np.log10(c.max()), 200)
        fx = model_frac(d["model"], xs, p)
        curves = [fx] if norm else [ru_r[r] + fx * amp[r] for r in range(n_rep)]
        for cv in curves:
            ax.plot(xs, cv, color="black", linewidth=1.2, linestyle=(0, (3, 2)), zorder=2)
            px.append(xs); py.append(cv)
    ax.set_xscale("log")
    _decades(ax)
    ax.set_ylabel(Y_LABEL_FRACTION if norm else Y_RAW, fontsize=FONT_SIZE, labelpad=1)
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f" if norm else f"%.{Y_DECIMALS}f"))
    if norm:
        _norm_ylim(ax, yy[pos])
        ax.yaxis.set_major_locator(_FRAC_TICKS)
    if SHOW_PANEL_TITLES:
        ax.set_title(name, fontsize=TITLE_FONT_SIZE, fontweight="bold", pad=3)
    ax.tick_params(labelsize=TICK_FONT_SIZE, length=3, width=0.5, pad=1.5)
    ax.tick_params(which="minor", length=1.5, width=0.3)
    for sp in ax.spines.values():
        sp.set_linewidth(0.5)
    # v17 puts the box bottom-left; for rising curves bottom-right is the empty corner
    # (fraction bound always rises, so its empty corner is top-left)
    rising = norm or np.mean(amp) > 0
    lx, ly, va, ha = ((0.05, 0.95, "top", "left") if norm else
                      (0.95 if rising else 0.05, 0.05, "bottom", "right" if rising else "left"))
    label = ax.text(lx, ly, _kd_text(res), transform=ax.transAxes,
            fontsize=FONT_SIZE, va=va, ha=ha,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="none"))

    target = ax
    if ax_res is not None:
        ax_res.set_xscale("log")
        if show_fit:
            rr = d["resid"] / amp[rep] if norm else d["resid"]
            ax_res.scatter(c[pos], rr[pos], s=8, color=MARKER_COLOR, alpha=0.7, marker="x", linewidths=0.8)
            ax_res.axhline(0, color="0.5", lw=0.5)
        else:
            ax_res.text(0.5, 0.5, "no fit", transform=ax_res.transAxes,
                        ha="center", va="center", color="0.5", fontsize=TICK_FONT_SIZE + 1.5)
            ax_res.set_yticks([])
        ax_res.set_ylabel("Resid.", fontsize=TICK_FONT_SIZE + 1.5)
        _decades(ax_res)
        ax_res.tick_params(labelsize=max(TICK_FONT_SIZE - 1, 4), length=2, width=0.5, pad=2)
        for sp in ax_res.spines.values():
            sp.set_linewidth(0.5)
        ax.tick_params(labelbottom=False)
        target = ax_res
    target.set_xlabel(X_LABEL, fontsize=FONT_SIZE, labelpad=1, loc="left")
    return ax, label, np.concatenate(px), np.concatenate(py)




def _declutter(fig, drawn, max_iter=12):
    """Move the y-axis floor (or ceiling, for a box at the top) until the Kd/R2 box
    overlaps no point and no curve."""
    for _ in range(max_iter):
        fig.canvas.draw()
        rend = fig.canvas.get_renderer()
        moved = False
        for ax, label, x, y in drawn:
            patch = label.get_bbox_patch()
            bb = (patch or label).get_window_extent(rend).expanded(1.04, 1.12)
            pts = ax.transData.transform(np.column_stack([x, y]))
            hit = ((pts[:, 0] >= bb.x0) & (pts[:, 0] <= bb.x1) &
                   (pts[:, 1] >= bb.y0) & (pts[:, 1] <= bb.y1))
            if hit.any():
                lo, hi = ax.get_ylim()
                if label.get_va() == "top":            # box at the top: raise the ceiling
                    ax.set_ylim(lo, hi + 0.12 * (hi - lo))
                else:
                    ax.set_ylim(lo - 0.12 * (hi - lo), hi)
                moved = True
        if not moved:
            break
    for ax, *_ in drawn:                               # enough decimals that no two y labels repeat
        if ax.get_ylabel() != Y_RAW:
            continue
        ticks = ax.get_yticks()
        step = np.min(np.diff(ticks)) if len(ticks) > 1 else 1
        dec = max(Y_DECIMALS, int(np.ceil(-np.log10(step) - 1e-9)))
        ax.yaxis.set_major_formatter(FormatStrFormatter(f"%.{dec}f"))


def _tiles(items, ncol, residuals):
    """Fixed 2.4 x 1.75 in tiles, no gaps between them, each laid out as its own
    small figure - the same geometry as the v17 PowerPoint panels."""
    nrow = int(np.ceil(len(items) / ncol))
    ph = PANEL_H + (0.5 if residuals else 0)
    fig = plt.figure(figsize=(PANEL_W * ncol, ph * nrow), layout="constrained")
    fig.get_layout_engine().set(w_pad=0.03, h_pad=0.03, wspace=0, hspace=0)
    subs = fig.subfigures(nrow, ncol, wspace=0, hspace=0, squeeze=False)
    drawn = []
    for i, (name, d, res) in enumerate(items):
        sf = subs[divmod(i, ncol)]
        if residuals:
            ax, axr = sf.subplots(2, 1, sharex=True, gridspec_kw=dict(height_ratios=[3, 1]))
        else:
            ax, axr = sf.subplots(), None
        drawn.append(draw(ax, axr, name, d, res))
    _declutter(fig, drawn)
    return fig


def _sorted(items):
    if FIT_CRITERIA == "strict":                     # determined Kds ascending, N.D. last
        return sorted(items, key=lambda it: (not it[2]["Kd_determined"], it[2]["Kd_nM"]))
    # v17: R2 >= 0.75 by Kd (tightest first), then the rest by R2 (worst last)
    def key(it):
        r = it[2]
        good = r["Kd_determined"] and r["R2"] >= 0.75
        return (0, r["Kd_nM"]) if good else (1, -r["R2"])
    return sorted(items, key=key)


def _fixed_box_axes(box_w, box_h, ml=None, mr=None, mb=None, mt=None):
    """A figure with a fixed box_w x box_h inch axes BOX inside fixed label margins, so the
    canvas is a fixed size with the box in a fixed place - panels line up whatever the
    y-label width. (Saved WITHOUT bbox_inches='tight'.)"""
    from mpl_toolkits.axes_grid1 import Divider, Size
    ml = PANEL_MARGIN_L if ml is None else ml
    mr = PANEL_MARGIN_R if mr is None else mr
    mb = PANEL_MARGIN_B if mb is None else mb
    mt = PANEL_MARGIN_T if mt is None else mt
    fig = plt.figure(figsize=(ml + box_w + mr, mb + box_h + mt))
    h = [Size.Fixed(ml), Size.Fixed(box_w), Size.Fixed(mr)]
    v = [Size.Fixed(mb), Size.Fixed(box_h), Size.Fixed(mt)]
    div = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)
    ax = fig.add_axes(div.get_position(), axes_locator=div.new_locator(nx=1, ny=1))
    return fig, ax


def plot_sample(name, d, res, path):
    if SHOW_RESIDUALS:                                 # residual panels keep the grid layout
        fig = _tiles([(name, d, res)], 1, True)
        fig.savefig(path + ".png", dpi=FIG_DPI)
        fig.savefig(path + ".svg")
        plt.close(fig)
        return
    fig, ax = _fixed_box_axes(PANEL_BOX_W, PANEL_BOX_H)
    drawn = [draw(ax, None, name, d, res)]
    _declutter(fig, drawn)
    fig.savefig(path + ".png", dpi=FIG_DPI)
    fig.savefig(path + ".svg")
    plt.close(fig)


def plot_grid(items, path, ncol=3):
    fig = _tiles(_sorted(items), ncol, SHOW_RESIDUALS)
    for ext in ("png", "svg", "pdf"):
        fig.savefig(f"{path}.{ext}", dpi=FIG_DPI)
    plt.show() if "ipykernel" in sys.modules else None
    plt.close(fig)


# ----------------------------------------------------------------------------
# Overlay - all chosen samples on one axis, mean +/- SD across repeats
# ----------------------------------------------------------------------------

OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00",
             "#56B4E9", "#000000", "#F0E442"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "h"]
CONC_TOL = 0.01                                  # concentrations within 1% are the same point


def _group_conc(c):
    """Group positive concentrations that agree within CONC_TOL. Returns a group
    index per point (-1 for 0 nM) and the mean concentration of each group."""
    g = np.full(c.size, -1)
    order = np.argsort(c)
    gid, start, centres, members = -1, None, [], []
    for i in order:
        if c[i] <= 0:
            continue
        if start is None or c[i] > start * (1 + CONC_TOL):
            gid += 1; start = c[i]; members.append([])
        g[i] = gid; members[gid].append(c[i])
    return g, np.array([np.mean(m) for m in members])


def overlay_points(d, norm):
    """Mean, SD and n per concentration across repeats (n counts repeats; a repeat
    with two wells at one concentration contributes their mean)."""
    c, y, rep, p = d["c"], d["y"], d["rep"], d["p"]
    ru, rb = d["ru"], d["rb"]
    yy = (y - ru[rep]) / (rb - ru)[rep] if norm else y
    g, xs = _group_conc(c)
    mean, sd, n = [], [], []
    for k in range(xs.size):
        per_rep = [yy[(g == k) & (rep == r)].mean() for r in np.unique(rep[g == k])]
        mean.append(np.mean(per_rep)); n.append(len(per_rep))
        sd.append(np.std(per_rep, ddof=1) if len(per_rep) > 1 else np.nan)
    return xs, np.array(mean), np.array(sd), np.array(n)


def _val_pm(v, lo, hi):
    """'79.6 ± 12 nM' (or '2.73 ± 0.22 µM'); KD_LABEL = "ci" gives '79.6 (63–93) nM'."""
    sc, unit = (1000, "µM") if v >= 1000 else (1, "nM")
    out = f"{float(f'{v / sc:.3g}'):g}"
    if np.isfinite(lo) and np.isfinite(hi):
        if KD_LABEL == "ci":
            out += f" ({float(f'{lo / sc:.3g}'):g}–{float(f'{hi / sc:.3g}'):g})"
        else:
            out += f" ± {float(f'{(hi - lo) / 2 / sc:.2g}'):g}"
    return f"{out} {unit}"


def _overlay_label(name, res, detail=False):
    """Legend entry. detail=True (panel-sized overlays): name / Kd ± half 95% CI (or EC50 ±,
    then n_H) / n repeats, R² - each optional line switched in LABELS."""
    hill = res.get("model") == "hill"
    sym = "~" if res.get("Kd_approx") else "="
    lo, hi = res["Kd_CI95_low"], res["Kd_CI95_high"]
    key = EC50_SYMBOL if hill else (KD1_SYMBOL if res.get("model") == "two_site" else KD_SYMBOL)
    if not hill and res["Kd_nM"] < T / 2:          # below the resolution limit
        ub = hi if np.isfinite(hi) else T / 2
        val, sym = f"{ub:.3g} nM", "<"
    elif detail:
        val = _val_pm(res["Kd_nM"], lo, hi)
    else:
        val = _val_pm(res["Kd_nM"], np.nan, np.nan)
    if not detail:
        extra = (f", {NH_SYMBOL} = {res['hill_n']:.2f}" if hill else
                 (f", n={res['n_repeats']}" if LEGEND_SHOW_N else ""))
        return f"{name} ({key} {sym} {val}{extra})"
    # detail: "K11 (K_D = 187 ± 50 nM)" then "n = 5, R² = 0.991" (Hill: n_H first)
    second = ([f"{NH_SYMBOL} = {res['hill_n']:.2f}"] if hill else []) + \
             ([f"n = {res['n_repeats']}"] if LEGEND_SHOW_N else []) + \
             ([f"R² = {res['R2']:.3f}"] if LEGEND_SHOW_R2 else [])
    first = f"{name} ({key} {sym} {val})"
    return first + ("\n" + ", ".join(second) if second else "")


def _overlay_items(items):
    ok = [it for it in items if it[2]["Kd_determined"]]
    if OVERLAY_SAMPLES is None:
        return sorted(ok, key=lambda it: it[2]["Kd_nM"])
    by_name = {it[0]: it for it in ok}
    all_names = {it[0] for it in items}
    out = []
    for nm in OVERLAY_SAMPLES:
        if nm in by_name:
            out.append(by_name[nm])
        elif nm in all_names:
            print(f"Overlay: {nm} has no determined Kd - left out")
        else:
            print(f"Overlay: no sample called {nm!r}")
    return out


def _row_geometry():
    """Panel geometry for the figure rows (D/E style): PANELS_PER_ROW panels fill
    PAGE_WIDTH_MM exactly, every data box the same size. Returns inches."""
    pw = PAGE_WIDTH_MM / 25.4 / len(ROW_PANELS)          # one panel's canvas width
    bw = pw - ROW_MARGIN_L - ROW_MARGIN_R
    bh = bw / ROW_BOX_ASPECT
    return pw, bw, bh


def _legend_clear(fig, ax, leg, max_iter=15):
    """Raise the y-axis ceiling until the legend overlaps no data point or curve
    (fraction-bound ticks stop at 1.0, so the headroom stays blank)."""
    if leg is None:
        return
    pts = []
    for ln in ax.lines:
        x, y = ln.get_xdata(), ln.get_ydata()
        if len(x) > 2 and len(x) == len(y):
            pts.append(np.column_stack([np.asarray(x, float), np.asarray(y, float)]))
    for coll in ax.collections:
        for path in coll.get_paths():
            v = path.vertices
            if len(v):
                pts.append(v)
    if not pts:
        return
    pts = np.concatenate(pts)
    pts = pts[np.isfinite(pts).all(axis=1) & (pts[:, 0] > 0)]
    for _ in range(max_iter):
        fig.canvas.draw()
        bb = leg.get_window_extent(fig.canvas.get_renderer()).expanded(1.02, 1.06)
        d = ax.transData.transform(pts)
        if not ((d[:, 0] >= bb.x0) & (d[:, 0] <= bb.x1) & (d[:, 1] >= bb.y0) & (d[:, 1] <= bb.y1)).any():
            break
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi + 0.08 * (hi - lo))


def draw_overlay(ax, items, norm, legend=True):
    """All chosen samples on one axes: points = mean +/- SD across repeats, curve = global
    fit, shaded 95% bootstrap band, Kd tick on the x-axis. Returns the legend (or None)."""
    chosen = _overlay_items(items)
    if not chosen:
        return None
    if len(chosen) > len(OKABE_ITO):
        print(f"Overlay: {len(chosen)} samples - colours repeat beyond {len(OKABE_ITO)}; "
              f"consider OVERLAY_SAMPLES")
    order, plotted = [], []
    for i, (name, d, res) in enumerate(chosen):
        col, mk = OKABE_ITO[i % len(OKABE_ITO)], MARKERS[i % len(MARKERS)]
        lab = _overlay_label(name, res, detail=OVERLAY_LEGEND_DETAIL)
        order.append(lab)
        ru, rb = d["ru"], d["rb"]
        x, m, sd, n = overlay_points(d, norm)
        ru_m, amp_m = ru.mean(), (rb - ru).mean()          # raw: mean of the repeat curves
        multi = n > 1
        ax.errorbar(x[multi], m[multi], yerr=sd[multi], fmt=mk, ms=MARKER_PT, color=col,
                    mfc=col, mec=col, elinewidth=0.7, capsize=2, capthick=0.7,
                    zorder=3, label=lab if multi.any() else None)
        if (~multi).any():                                 # single-repeat points: hollow
            ax.plot(x[~multi], m[~multi], mk, ms=MARKER_PT, mfc=("none" if TRANSPARENT else "white"),
                    mec=col, mew=0.8, zorder=3, label=None if multi.any() else lab)
        plotted += [m, m + np.nan_to_num(sd), m - np.nan_to_num(sd)]
        xs = np.logspace(np.log10(x.min()), np.log10(d["c"].max()), 200)
        scale = (lambda f: f) if norm else (lambda f, a=ru_m, b=amp_m: a + f * b)
        ax.plot(xs, scale(model_frac(d["model"], xs, d["p"])), color=col, lw=1.2, zorder=2)
        if OVERLAY_CI_BAND and d.get("boot") is not None:
            fb = np.array([model_frac(d["model"], xs, pb) for pb in d["boot"]])
            lo_b, hi_b = np.percentile(fb, [2.5, 97.5], axis=0)
            ax.fill_between(xs, scale(lo_b), scale(hi_b), color=col, alpha=0.15, lw=0, zorder=1)
        if OVERLAY_KD_TICKS and res["Kd_nM"] >= T / 2:
            ax.axvline(res["Kd_nM"], ymin=0, ymax=0.05, color=col, lw=1.5, zorder=4)
    ax.set_xscale("log")
    _decades(ax)
    ax.set_xlabel(X_LABEL, fontsize=FONT_SIZE, labelpad=1)
    ax.set_ylabel(Y_LABEL_FRACTION if norm else Y_RAW, fontsize=FONT_SIZE, labelpad=1)
    if norm:
        _norm_ylim(ax, np.concatenate(plotted), -0.12, 1.12)
        ax.yaxis.set_major_locator(_FRAC_TICKS)
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    else:                                               # 1/2/5 steps, so ticks are never rounded
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 5, 10]))
        step = np.diff(ax.get_yticks()).min() if len(ax.get_yticks()) > 1 else 1
        dec = max(0, int(np.ceil(-np.log10(step) - 1e-9)))
        ax.yaxis.set_major_formatter(FormatStrFormatter(f"%.{dec}f"))
    ax.tick_params(labelsize=TICK_FONT_SIZE, length=3, width=0.5, pad=1.5)
    ax.tick_params(which="minor", length=1.5, width=0.3)
    for sp in ax.spines.values():
        sp.set_linewidth(0.5)
    if not legend or not (norm or not OVERLAY_LEGEND_NORM_ONLY):
        return None
    h, l = ax.get_legend_handles_labels()                  # matplotlib puts errorbars last: re-sort
    hl = sorted(zip(h, l), key=lambda t: order.index(t[1]) if t[1] in order else 99)
    return ax.legend([t[0] for t in hl], [t[1] for t in hl], fontsize=LEGEND_FONT_SIZE,
                     frameon=False, loc="upper left", borderaxespad=0.3, handletextpad=0.3,
                     labelspacing=0.35, borderpad=0.2)


def plot_overlay(items, path, axis=None, panel=False, show=True):
    """One overlay as its own file. panel=True: the figure-row panel size (same box as in
    Figure_rows), so single panels and the assembled rows match exactly."""
    norm = (axis or OVERLAY_AXIS) == "normalised"
    if not _overlay_items(items):
        print("Overlay: nothing to plot")
        return None
    if panel:
        pw, bw, bh = _row_geometry()
        fig, ax = _fixed_box_axes(bw, bh, ROW_MARGIN_L, ROW_MARGIN_R, ROW_MARGIN_B, ROW_MARGIN_T)
    else:
        fig, ax = plt.subplots(figsize=(5.2, 3.0), layout="constrained")
    leg = draw_overlay(ax, items, norm)
    _legend_clear(fig, ax, leg)
    for ext in ("png", "svg", "pdf"):
        fig.savefig(f"{path}.{ext}", dpi=FIG_DPI)
    if show and "ipykernel" in sys.modules:
        plt.show()
    plt.close(fig)
    return f"{path}.png"


def plot_rows(rows, path, show=True):
    """D/E-style figure: one row per readout (list of (readout, items)), panels per row
    from ROW_PANELS, all boxes identical, total width PAGE_WIDTH_MM. Place at 100%."""
    from mpl_toolkits.axes_grid1 import Divider, Size
    rows = [(r, it) for r, it in rows if _overlay_items(it)]
    if not rows:
        print("Figure rows: nothing to plot")
        return None
    pw, bw, bh = _row_geometry()
    ncol, nrow = len(ROW_PANELS), len(rows)
    rh = ROW_MARGIN_B + bh + ROW_MARGIN_T
    fig = plt.figure(figsize=(pw * ncol, rh * nrow))
    hs = []
    for _ in range(ncol):
        hs += [Size.Fixed(ROW_MARGIN_L), Size.Fixed(bw), Size.Fixed(ROW_MARGIN_R)]
    vs = []
    for _ in range(nrow):
        vs += [Size.Fixed(ROW_MARGIN_B), Size.Fixed(bh), Size.Fixed(ROW_MARGIN_T)]
    div = Divider(fig, (0, 0, 1, 1), hs, vs, aspect=False)
    for ri, (r, items) in enumerate(rows):
        set_readout(r)
        for ci, kind in enumerate(ROW_PANELS):
            ax = fig.add_axes(div.get_position(),
                              axes_locator=div.new_locator(nx=1 + 3 * ci, ny=1 + 3 * (nrow - 1 - ri)))
            leg = draw_overlay(ax, items, kind == "normalised")
            _legend_clear(fig, ax, leg)
    for ext in ("png", "svg", "pdf"):
        fig.savefig(f"{path}.{ext}", dpi=FIG_DPI)
    if show and "ipykernel" in sys.modules:
        plt.show()
    plt.close(fig)
    return f"{path}.png"


def export_rows_pptx(pngs, path):
    """Figure rows at true size (pixels / FIG_DPI) on an A4 portrait slide, stacked."""
    try:
        from pptx import Presentation
        from pptx.util import Inches
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "python-pptx", "-q"])
        from pptx import Presentation
        from pptx.util import Inches
    from PIL import Image
    pngs = [p for p in pngs if p and os.path.exists(p)]
    if not pngs:
        return
    prs = Presentation()
    prs.slide_width, prs.slide_height = Inches(210 / 25.4), Inches(297 / 25.4)
    left = (210 - PAGE_WIDTH_MM) / 2 / 25.4
    for png in pngs:                                   # one figure per slide
        slide = prs.slides.add_slide(prs.slide_layouts[6])
        w, h = Image.open(png).size
        slide.shapes.add_picture(png, Inches(left), Inches(0.6),
                                 width=Inches(w / FIG_DPI), height=Inches(h / FIG_DPI))
    prs.save(path)


def _pptx_panels(prs, items, Inches):
    """v17 grid: 3 x 6 panels of PANEL_W x PANEL_H in per portrait slide."""
    per_slide = 3 * 6
    items = _sorted(items)
    for s0 in range(0, len(items), per_slide):
        slide = prs.slides.add_slide(prs.slide_layouts[6])
        for i, item in enumerate(items[s0:s0 + per_slide]):
            r, col = divmod(i, 3)
            fig = _tiles([item], 1, residuals=False)
            buf = BytesIO()
            fig.savefig(buf, format="png", dpi=300)
            plt.close(fig)
            buf.seek(0)
            slide.shapes.add_picture(buf, Inches(0.15 + col * PANEL_W), Inches(0.05 + r * (PANEL_H + 0.03)),
                                     width=Inches(PANEL_W), height=Inches(PANEL_H))


def _pptx_native_panels(prs, pngs, Inches):
    """Panel-sized figures at their true size (pixels / FIG_DPI), two per row, so every
    box is the same physical size on the slide - ready to assemble into a figure."""
    from PIL import Image
    pngs = [p for p in pngs if p and os.path.exists(p)]
    if not pngs:
        return
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    ncol = max(1, int((prs.slide_width / 914400 - 0.3) //
                      (Image.open(pngs[0]).size[0] / FIG_DPI + 0.3)))
    for k, png in enumerate(pngs):
        w, h = Image.open(png).size
        wi, hi = w / FIG_DPI, h / FIG_DPI
        r, col = divmod(k, ncol)
        slide.shapes.add_picture(png, Inches(0.3 + col * (wi + 0.3)), Inches(0.3 + r * (hi + 0.2)),
                                 width=Inches(wi), height=Inches(hi))


def export_pptx(items, path, overlay_png=None, extra_pngs=(), sections=(), panel_pngs=()):
    """v17 layout: portrait 7.5 x 10.83 in slides, 3 x 6 panels of 2.4 x 1.75 in at 300 dpi.
    Panels are pictures; residual panels are never included here."""
    try:
        from pptx import Presentation
        from pptx.util import Inches
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "python-pptx", "-q"])
        from pptx import Presentation
        from pptx.util import Inches
    prs = Presentation()
    prs.slide_width, prs.slide_height = Inches(7.5), Inches(10.83)
    from PIL import Image
    for sec_items, sec_pngs, sec_panels in [(items, [overlay_png, *extra_pngs], panel_pngs),
                                            *sections]:
        _pptx_panels(prs, sec_items, Inches)
        _pptx_native_panels(prs, sec_panels, Inches)
        for png in sec_pngs:                          # one full-width figure per slide
            if not png or not os.path.exists(png):
                continue
            w, h = Image.open(png).size
            slide = prs.slides.add_slide(prs.slide_layouts[6])
            width = 7.0
            slide.shapes.add_picture(png, Inches(0.25), Inches(0.3),
                                     width=Inches(width), height=Inches(width * h / w))
    prs.save(path)


# ----------------------------------------------------------------------------
# Methods text
# ----------------------------------------------------------------------------

# ----------------------------------------------------------------------------
# Hill figure set (HILL_FIGURES) - descriptive alternative, never the reported Kd
# ----------------------------------------------------------------------------

def hill_items(items):
    """Re-express each (name, d, res) with the Hill fit, so the normal plotting code
    draws Hill curves and EC50 / n_H labels."""
    out = []
    for name, d, res in items:
        h = d.get("hill")
        if h is None:
            continue
        dh = dict(d, model="hill", p=h["p"], resid=h["resid"], ru=h["ru"], rb=h["rb"],
                  kd=10 ** h["p"][0], boot=h["boot"])
        rh = dict(res, model="hill", Kd_nM=res["hill_EC50_nM"],
                  Kd_CI95_low=res.get("hill_EC50_CI95_low", np.nan),
                  Kd_CI95_high=res.get("hill_EC50_CI95_high", np.nan),
                  Kd_approx=False, R2=h["R2"],
                  # judged on its own fit, not the reported model's
                  Kd_determined=bool(np.isfinite(res["hill_EC50_nM"]) and
                                     res["hill_EC50_nM"] < 0.99 * d["c"].max() * 100 and
                                     h["R2"] >= SCREEN_MIN_R2))
        out.append((name, dh, rh))
    return out


def plot_hill_linear(items, path):
    """Classic Hill plot: log10(theta / (1 - theta)) vs log concentration, theta from each
    repeat's own Hill baselines, mean across repeats per concentration. The line has
    slope n_H and crosses zero at EC50. Points with theta outside 0.03-0.97 are left
    out (the transform explodes there). Total concentration is used for the x-axis
    (free ~ total when the tracer is far below EC50)."""
    fig, ax = plt.subplots(figsize=(5.2, 3.0), layout="constrained")
    order = []
    for i, (name, d, res) in enumerate(sorted(items, key=lambda it: it[2]["Kd_nM"])):
        col, mk = OKABE_ITO[i % len(OKABE_ITO)], MARKERS[i % len(MARKERS)]
        x, m, sd, n = overlay_points(d, True)
        ok = (m > 0.03) & (m < 0.97)
        yl = np.log10(m[ok] / (1 - m[ok]))
        lab = _overlay_label(name, res)
        order.append(lab)
        ax.plot(x[ok], yl, mk, ms=4, color=col, mec=col, zorder=3, label=lab)
        ec50, nh = res["Kd_nM"], res["hill_n"]
        xs = np.logspace(np.log10(x[ok].min()) - 0.3, np.log10(x[ok].max()) + 0.3, 50)
        ax.plot(xs, nh * np.log10(xs / ec50), color=col, lw=1.2, zorder=2)
    ax.axhline(0, color="0.6", lw=0.6, ls=(0, (3, 2)), zorder=1)
    ax.set_xscale("log")
    _decades(ax)
    ax.set_xlabel(X_LABEL, fontsize=FONT_SIZE, labelpad=1)
    ax.set_ylabel(Y_LABEL_HILL_PLOT, fontsize=FONT_SIZE, labelpad=1)
    ax.tick_params(labelsize=TICK_FONT_SIZE, length=3, width=0.5, pad=1.5)
    ax.tick_params(which="minor", length=1.5, width=0.3)
    for sp in ax.spines.values():
        sp.set_linewidth(0.5)
    ax.legend(fontsize=LEGEND_FONT_SIZE, frameon=False, loc="upper left",
              bbox_to_anchor=(1.01, 1), handletextpad=0.3, borderaxespad=0)
    for ext in ("png", "svg", "pdf"):
        fig.savefig(f"{path}.{ext}", dpi=FIG_DPI)
    if "ipykernel" in sys.modules:
        plt.show()
    plt.close(fig)


def panel_set(items, folder, prefix=""):
    """The two row panels as separate files (fraction bound, raw), same box as Figure_rows."""
    out = []
    for tag, axis in (("overlay_panel", "normalised"), ("overlay_raw", "raw")):
        png = plot_overlay(items, os.path.join(folder, prefix + tag), axis=axis, panel=True, show=False)
        if png:
            out.append(png)
    return out


def hill_figures(items, sub):
    hi = hill_items(items)
    if not hi:
        print("Hill figures: no Hill fits to plot")
        return None
    hsub = os.path.join(sub, "hill")
    os.makedirs(hsub, exist_ok=True)
    print(f"Hill figure set -> {hsub}/ (descriptive; EC50 and n_H are not binding constants)")
    for name, d, res in hi:
        safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in name)
        plot_sample(name, d, res, os.path.join(hsub, safe))
    plot_grid(hi, os.path.join(hsub, "all_samples"))
    ov = plot_overlay(hi, os.path.join(hsub, "overlay_hill"), axis="normalised")
    panels = panel_set(hi, hsub, "hill_")
    plot_hill_linear(hi, os.path.join(hsub, "hill_plot"))
    return hi, [ov, os.path.join(hsub, "hill_plot.png")], panels   # appended to the main deck


def _exclusion_sentence():
    out = ""
    if MAX_CONC_NM is not None:
        out += f" Wells above {MAX_CONC_NM / 1000:g} µM {TITRANT_NAME} were excluded from all fits."
    if DROP_ANOMALIES and _DROPPED:
        kinds = ", ".join(sorted(set(_DROPPED)))
        out += f" Wells flagged by the instrument software (n = {len(_DROPPED)}; {kinds}) were excluded."
    return out


def _criteria_sentence():
    if FIT_CRITERIA == "strict":
        return (f"Kd was reported as not determined (N.D.) when the signal-to-noise ratio "
                f"(mean amplitude / residual SD) was below {MIN_SIGNAL_TO_NOISE:g} or the 95% CI "
                f"spanned more than {MAX_CI_FOLD:g}-fold.")
    return (f"Kd was reported as not determined (N.D.) for fits with R² < {SCREEN_MIN_R2:g}, and "
            f"as approximate (~) for fits with R² < {SCREEN_APPROX_R2:g} or less than 50% bound "
            f"at the highest concentration.")


def _overlay_sentence():
    if not OVERLAY:
        return ""
    if OVERLAY_AXIS == "normalised":
        pts = ("each repeat was converted to fraction bound using its own fitted baselines, "
               "and points show the mean +/- SD across repeats")
    else:
        pts = ("points show the mean +/- SD of the raw signal across repeats (the SD "
               "includes baseline differences between repeats)")
    band = (" Shaded bands show the 95% bootstrap confidence band of each fit."
            if OVERLAY_CI_BAND else "")
    return (f" In overlay plots, {pts} at each concentration; points measured in a "
            f"single repeat are shown without error bars. Error bars are for display "
            f"only: all fits used every individual data point.{band}")


def _wrap(text, width=80):
    import textwrap
    out = []
    for para in text.split("\n\n"):
        if para.startswith("  "):                      # equation block: keep as is
            out.append(para)
        else:
            out.append(textwrap.fill(" ".join(para.split()), width,
                                     break_on_hyphens=False, break_long_words=False))
    return "\n\n".join(out) + "\n"


def methods_text(table=None):
    if READOUT == "tric":
        S = "F"
        what = ("normalised fluorescence (Fnorm, per mil) from temperature-related intensity "
                "change (TRIC) measurements")
    else:
        S = "R"
        what = "670/650 nm emission ratio (R)"
    multi = table is not None and len(table) and table["n_repeats"].max() > 1
    pm = ("half the width of the 95% confidence interval" if KD_LABEL == "plus_minus"
          else None)
    if multi:
        reps = ("Independent repeats were fitted globally with a shared Kd and repeat-specific "
                f"{S}_unbound and {S}_bound. ")
        boot = "residuals resampled within each repeat"
        each = (" Each repeat was also fitted separately, and the mean +/- SD of these Kd "
                "values is reported.")
    else:
        reps = "Replicate measurements of each sample were pooled into a single fit. "
        boot = "residuals resampled with replacement"
        each = ""
    return _wrap(f"""Data analysis. For each sample, the {what} was fitted
as a function of total {TITRANT_NAME} concentration (C) to a 1:1 binding model
that accounts for ligand depletion (Morrison quadratic):

  {S} = {S}_unbound + ({S}_bound - {S}_unbound) * f,
  f = [(C + T + Kd) - sqrt((C + T + Kd)^2 - 4*C*T)] / (2*T)

where T is the total concentration of tracer ({TRACER_DESCRIPTION}), labelled plus
unlabelled ({T:g} nM in the well; {TRACER_DYE_NM:g} nM dye). {reps}Fitting used
non-linear least squares (SciPy curve_fit), with Kd fitted on a log10 scale and
constrained to between 1/100 of the lowest and 100x the highest non-zero
concentration; tracer-only (0 nM) wells were included. Fits reaching the upper
Kd limit were reported as not determined. The 95% confidence interval for Kd was
obtained by residual bootstrap ({N_BOOTSTRAP} resamples, {boot};
2.5th-97.5th percentiles).{each}{" Values given as Kd +/- x report x as " + pm + "." if pm else ""}
R2 was calculated as 1 - SS_res/SS_tot, with SS_tot taken about the mean of each
repeat. Because the
midpoint of the curve lies at Kd + T/2, Kd values below T/2 ({T/2:.3g} nM) are not
resolved and are reported as upper bounds, given by the upper 95% confidence limit.
{_criteria_sentence()}{_overlay_sentence()}{_exclusion_sentence()} Models including a linear non-specific term, a Hill
coefficient, or a second independent class of site were compared with the reported model
by AICc as diagnostics only.
""" + (_twosite_methods(S) if FIT_MODEL == "two_site" else "")
       + (_hill_methods(S) if HILL_FIGURES else "")
       + f"\nAnalysis: Dianthus analysis notebook {NOTEBOOK_VERSION} "
         f"(https://github.com/eoinleen/dianthus-analysis/tree/{NOTEBOOK_VERSION}).\n")


def _hill_methods(S):
    return "\n" + _wrap(f"""Hill analysis (descriptive). For comparison, the same data were fitted with
a Hill equation, {S} = {S}_unbound + ({S}_bound - {S}_unbound) / (1 + (EC50/L)^nH), with
EC50 and nH shared between repeats and the free ligand concentration L obtained from
mass balance. The 95% CIs of EC50 and nH were obtained by the same residual bootstrap.
EC50 and nH describe the shape of the titration and are not interpreted as an
equilibrium dissociation constant or as cooperativity.
""")


def _twosite_methods(S):
    kd2 = (f"Kd2 was fixed at {TWO_SITE_KD2_NM:g} nM. " if TWO_SITE_KD2_NM else
           "Kd2 was constrained to be at least 3-fold weaker than Kd1. ")
    return "\n" + _wrap(f"""Two-site model. Data were fitted with two independent classes of
site, each described by the depletion-corrected expression above:

  {S} = {S}_unbound + ({S}_bound - {S}_unbound) * [a * f(Kd1) + (1 - a) * f(Kd2)]

where a is the fraction of the total signal change from the high-affinity site
(shared between repeats, as were Kd1 and Kd2). {kd2}The reported Kd (and its 95%
bootstrap CI) is Kd1.
""")


# ----------------------------------------------------------------------------
# Run
# ----------------------------------------------------------------------------

def _process_file(samples, label, sub, readout):
    set_readout(readout)
    if SAMPLE_LABELS:                                 # display names (LABELS); same name = pooled
        renamed = {}
        for k, v in samples.items():
            renamed.setdefault(SAMPLE_LABELS.get(k, k), []).extend(v)
        samples = renamed
    print(f"\n=== {label}  ->  {Y_RAW}  |  model: {FIT_MODEL} ===")
    print(f"Total target in well: {T:g} nM (dye {TRACER_DYE_NM:g} nM) | resolution Kd ~ {T/2:.3g} nM")
    rows, items = [], []
    for name, reps in samples.items():
        try:
            res, d = analyse_sample(name, reps)
        except Exception as e:
            print(f"{name}: fit failed ({e})")
            continue
        rows.append(res)
        if not res["Kd_determined"] and BAD_FIT_PLOT == "skip":
            print(f"{name}: bad fit, left out of figures")
            continue
        items.append((name, d, res))
        safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in name)
        plot_sample(name, d, res, os.path.join(sub, safe))
    table = pd.DataFrame(rows)
    # Human-readable Kd column: N.D. for bad fits (numbers kept in Kd_nM for inspection)
    table.insert(3, "Kd_reported", [
        (f"< {(r['Kd_CI95_high'] if np.isfinite(r['Kd_CI95_high']) else T / 2):.3g}"
         if r["Kd_nM"] < T / 2 else ("~ " if r["Kd_approx"] else "") + f"{r['Kd_nM']:.3g}")
        if r["Kd_determined"] else "N.D."
        for _, r in table.iterrows()])
    table.insert(1, "readout", readout)
    table.to_csv(os.path.join(sub, "results.csv"), index=False)
    if items:
        plot_grid(items, os.path.join(sub, "all_samples"))
        ov = plot_overlay(items, os.path.join(sub, OVERLAY_FILENAME)) if OVERLAY else None
        panels = panel_set(items, sub) if OVERLAY and OVERLAY_RAW_PANEL else []
        hill_sec = hill_figures(items, sub) if HILL_FIGURES else None
        if EXPORT_PPTX:                               # Hill slides (if any) follow the main ones
            export_pptx(items, os.path.join(sub, PPTX_FILENAME), ov, panel_pngs=panels,
                        sections=[hill_sec] if hill_sec else ())
    with open(os.path.join(sub, "methods.txt"), "w") as fh:
        fh.write(methods_text(table))

    show = ["sample", "n_repeats", "Kd_reported", "Kd_CI95_low", "Kd_CI95_high", "Kd2_nM",
            "frac_tight_site", "Kd_repeats_mean_nM", "Kd_repeats_SD_nM", "frac_bound_at_top",
            "hill_EC50_nM", "hill_n", "dAICc_1to1", "dAICc_twosite", "dAICc_hill", "flags"]
    with pd.option_context("display.max_colwidth", 80, "display.width", 200,
                           "display.float_format", "{:.3g}".format):
        print(table[[s for s in show if s in table.columns]].to_string(index=False))
    return items, table


def run(paths=None):
    if paths is None:
        paths = PATHS if "PATHS" in globals() else None
    if isinstance(paths, str):
        paths = [paths]
    paths = [p for p in (paths or []) if p]
    if not paths:
        print("No input files. Set PATHS (or upload) and run again.")
        return {}
    global T
    T = float(TOTAL_TARGET_NM)
    _DROPPED.clear()
    print(f"Dianthus analysis notebook {NOTEBOOK_VERSION}")                         # re-read settings on every run
    _configure_models()
    assert FIT_MODEL in MODEL_KEY, f"FIT_MODEL must be one of {list(MODEL_KEY)}"
    if os.path.isdir(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR)
    # Wide sheets are analysed one file at a time. Dianthus export tables (one per
    # repeat) are pooled per readout, so repeats of a sample are fitted together.
    groups = []                                        # (label, samples, readout)
    pooled = {}
    for path in paths:
        samples = load(path)
        r = detect_readout(samples) if AUTO_READOUT else READOUT
        if is_dianthus_export(path):
            g = pooled.setdefault(r, {})
            for name, reps in samples.items():
                g.setdefault(name, []).extend(reps)
        else:
            groups.append((os.path.basename(path), samples, r))
    for r, samples in pooled.items():
        groups.append((f"Dianthus export tables ({sum(len(v) for v in samples.values())} repeats)",
                       samples, r))
    datasets, tables = {}, {}
    for label, samples, r in groups:
        if r in datasets:
            stem = re.sub(r"\W+", "_", os.path.splitext(label)[0])[:40]
            print(f"Note: a second '{r}' dataset; its outputs go in {r}_{stem}/, "
                  f"the comparison figure keeps the first.")
            sub = os.path.join(OUTPUT_DIR, f"{r}_{stem}")
        else:
            sub = os.path.join(OUTPUT_DIR, r)
        os.makedirs(sub, exist_ok=True)
        items, table = _process_file(samples, label, sub, r)
        datasets.setdefault(r, items)
        tables[label] = table
    if datasets:                                       # D/E-style rows: TRIC row, then ratio row
        order = [r for r in ("tric", "ratio") if r in datasets]
        figs = [plot_rows([(r, datasets[r]) for r in order], os.path.join(OUTPUT_DIR, "Figure_rows"))]
        if HILL_FIGURES:
            figs.append(plot_rows([(r, hill_items(datasets[r])) for r in order],
                                  os.path.join(OUTPUT_DIR, "Figure_rows_hill")))
        if EXPORT_PPTX:
            export_rows_pptx(figs, os.path.join(OUTPUT_DIR, ROWS_PPTX))
    shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
    return tables

In [ ]:
# Upload one or more Excel/CSV files (Colab) or set PATHS to local files, then run.
# Feed the TRIC and the spectral-shift file together - each one's readout is detected
# automatically, and Figure_rows (TRIC row + ratio row) is written on top of the per-file output.
PATHS = None   # e.g. ["tric_plate.xlsx", "ratio_plate.xlsx"] when running locally

try:
    from google.colab import files
    up = files.upload()
    PATHS = list(up.keys())
except ImportError:
    pass

results = run(PATHS)

try:
    files.download(OUTPUT_DIR + ".zip")
except NameError:
    print(f"Outputs in {OUTPUT_DIR}/ and {OUTPUT_DIR}.zip")
